# Pratica – Lezione 5: Ottimizzazione II — tecniche avanzate e regolarizzazione

In questa pratica lavori su due idee centrali della lezione: il momento come correzione alla discesa del gradiente puro, e la scelta dell'ottimizzatore giusto per addestrare più in fretta e generalizzare meglio.

## Cosa farai in questa pratica

- Implementi a mano, in NumPy, la regola di aggiornamento della discesa del gradiente con momento e la applichi su una valle stretta, un paesaggio di perdita mal condizionato.
- Addestri un perceptron multistrato (MLP, acronimo dell'inglese *multilayer perceptron*) in PyTorch su un dataset sintetico, confrontando tre ottimizzatori — discesa del gradiente stocastica (SGD, acronimo dell'inglese *stochastic gradient descent*) senza momento, SGD con momento e Adam — insieme al dropout come tecnica di regolarizzazione.

## Prerequisito

Prima di iniziare, assicurati di aver seguito la Lezione 5 in diretta (o di aver letto il capitolo "Ottimizzazione avanzata e regolarizzazione" del manuale). Devi capire cos'è il momento e perché il numero di condizionamento di un problema influisce sulla velocità di convergenza.

Esegui questo notebook in Google Colab. Non devi installare nulla: tutte le librerie che usi (NumPy, Matplotlib, PyTorch e scikit-learn) sono preinstallate.

## Esercizio 1 — Discesa con momento a mano, su una valle stretta

### Contesto

A lezione hai visto che la discesa del gradiente puro avanza lentamente in una valle stretta: una direzione ha molta più curvatura dell'altra, e il passo che evita di divergere nella direzione ripida risulta troppo piccolo per avanzare in fretta nella direzione dolce. Il momento corregge questo accumulando una "velocità" che smussa la lentezza e rafforza l'avanzamento costante.

### Compito

Completa la funzione `descenso_con_momento`, che implementa questa regola di aggiornamento (la stessa che PyTorch usa internamente):

```
v = beta * v + gradiente(punto_actual)
punto_actual = punto_actual - tasa_aprendizaje * v
```

Poi esegui la cella successiva, che applica la tua funzione a una funzione di valle stretta (una parabola con molta più curvatura su un asse che sull'altro) e confronta la traiettoria con quella della discesa del gradiente normale, già implementata.

### Suggerimenti

- `v` parte da zero e ha la stessa forma di `punto_inicial`.
- A ogni passo, prima aggiorni `v` con il gradiente del punto attuale e poi usi quel `v` per muovere il punto: non usare il gradiente direttamente per spostare il punto.
- Un valore tipico di `beta` è tra 0,9 e 0,99: più è alto, più la velocità ha memoria dei gradienti passati.
- Salva ogni punto attraversato nella lista `trayectoria`, come già fa `descenso_gradiente`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Valle estrecho: una parábola con mucha más curvatura en el eje x0 que en
# el eje x1. El cociente a / b es el número de condición del problema,
# igual que en la analogía del valle que viste en clase.
a, b = 100.0, 1.0


def perdida(punto):
    """Valor de la función de pérdida en un punto (x0, x1)."""
    x0, x1 = punto
    return 0.5 * (a * x0 ** 2 + b * x1 ** 2)


def gradiente(punto):
    """Gradiente de la función de pérdida en un punto (x0, x1)."""
    x0, x1 = punto
    return np.array([a * x0, b * x1])

In [ ]:
def descenso_gradiente(gradiente, punto_inicial, tasa_aprendizaje, n_pasos):
    """Descenso de gradiente normal. Devuelve la trayectoria completa,
    un array de forma (n_pasos + 1, 2), incluido el punto inicial."""
    punto_actual = np.array(punto_inicial, dtype=float)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        punto_actual = punto_actual - tasa_aprendizaje * gradiente(punto_actual)
        trayectoria.append(punto_actual.copy())

    return np.array(trayectoria)

In [ ]:
def descenso_con_momento(gradiente, punto_inicial, tasa_aprendizaje, beta, n_pasos):
    """Descenso de gradiente con momento. Devuelve la trayectoria completa,
    un array de forma (n_pasos + 1, 2), incluido el punto inicial."""
    punto_actual = np.array(punto_inicial, dtype=float)
    v = np.zeros_like(punto_actual)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # TODO: actualiza la velocidad acumulada con el gradiente del punto actual
        v = ...

        # TODO: actualiza el punto usando la velocidad que acabas de calcular
        punto_actual = ...

        trayectoria.append(punto_actual.copy())

    return np.array(trayectoria)

In [ ]:
punto_inicial = [1.0, 1.0]
tasa_aprendizaje = 0.0095
n_pasos = 100

trayectoria_gd = descenso_gradiente(gradiente, punto_inicial, tasa_aprendizaje, n_pasos)
trayectoria_momento = descenso_con_momento(
    gradiente, punto_inicial, tasa_aprendizaje, beta=0.9, n_pasos=n_pasos
)

print(f"Pérdida final, GD normal: {perdida(trayectoria_gd[-1]):.6f}")
print(f"Pérdida final, GD con momento: {perdida(trayectoria_momento[-1]):.6f}")

perdidas_gd = [perdida(p) for p in trayectoria_gd]
perdidas_momento = [perdida(p) for p in trayectoria_momento]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: contorno de la pérdida con las dos trayectorias superpuestas
x0_rango = np.linspace(-1.2, 1.2, 200)
x1_rango = np.linspace(-1.2, 1.2, 200)
malla_x0, malla_x1 = np.meshgrid(x0_rango, x1_rango)
malla_z = 0.5 * (a * malla_x0 ** 2 + b * malla_x1 ** 2)

ax1.contour(malla_x0, malla_x1, malla_z, levels=30, cmap="Greys", alpha=0.6)
ax1.plot(trayectoria_gd[:, 0], trayectoria_gd[:, 1], "o-", markersize=3, label="GD normal")
ax1.plot(trayectoria_momento[:, 0], trayectoria_momento[:, 1], "o-", markersize=3, label="GD con momento")
ax1.set_xlabel("x0 (curvatura alta)")
ax1.set_ylabel("x1 (curvatura baja)")
ax1.set_title("Trayectorias sobre el valle estrecho")
ax1.legend()

# Panel derecho: pérdida frente a iteración, en escala logarítmica
ax2.plot(perdidas_gd, label="GD normal")
ax2.plot(perdidas_momento, label="GD con momento")
ax2.set_yscale("log")
ax2.set_xlabel("Iteración")
ax2.set_ylabel("Pérdida (escala log)")
ax2.set_title("Velocidad de convergencia")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
def descenso_con_momento(gradiente, punto_inicial, tasa_aprendizaje, beta, n_pasos):
    """Discesa del gradiente con momento. Restituisce la traiettoria completa,
    un array di forma (n_pasos + 1, 2), incluso il punto iniziale."""
    punto_actual = np.array(punto_inicial, dtype=float)
    v = np.zeros_like(punto_actual)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # La velocità accumula il gradiente attuale sulla velocità precedente,
        # con decadimento esponenziale controllato da beta.
        v = beta * v + gradiente(punto_actual)

        # Il punto si muove in base alla velocità accumulata, non in base al
        # gradiente istantaneo: così si attenuano i cambi di direzione.
        punto_actual = punto_actual - tasa_aprendizaje * v

        trayectoria.append(punto_actual.copy())

    return np.array(trayectoria)
```

Con `beta=0.9` e lo stesso tasso di apprendimento della discesa del gradiente normale, il momento raggiunge una perdita finale di vari ordini di grandezza inferiore nello stesso numero di passi: la velocità accumulata compensa la lentezza nella direzione a bassa curvatura senza che tu debba rischiare un tasso di apprendimento più alto.
</details>

## Esercizio 2 — Scegliere il miglior ottimizzatore per addestrare in meno epoche

### Caso d'uso

Immagina di collaborare a un progetto di digitalizzazione con un'azienda della filiera Agro-Mar Alimentación: devi addestrare un classificatore semplice e il team tecnico ti chiede di raggiungere una buona precisione nel minor numero di epoche possibile, perché l'hardware disponibile è limitato. Prima di complicare l'architettura, la prima leva che hai è scegliere bene l'ottimizzatore.

### Compito

Addestri lo stesso perceptron multistrato (MLP) su un dataset sintetico non lineare (`make_moons`, di scikit-learn) con tre ottimizzatori diversi: SGD senza momento, SGD con momento e Adam. Il modello include uno strato `torch.nn.Dropout` per regolarizzare, come hai visto nella parte finale della lezione.

Completa l'istanziazione di ciascun ottimizzatore e definisci un valore di `weight_decay` (decadimento dei pesi) ragionevole. Alla fine, sovrapponi le tre curve di perdita per confrontare quale ottimizzatore converge più in fretta con lo stesso numero di epoche.

### Suggerimenti

- `torch.optim.SGD` accetta i parametri `lr`, `momentum` e `weight_decay`.
- `torch.optim.Adam` accetta `lr` e `weight_decay`; non ha bisogno di `momentum` perché incorpora già una logica di accumulo equivalente.
- Un valore tipico di `weight_decay` è tra 1e-5 e 1e-3: più è alto, più penalizzi i pesi grandi.
- Ricorda che `modelo.train()` attiva il dropout durante l'addestramento, già incluso nella funzione `entrenar`.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)

# --- Datos sintéticos no lineales ---
X_np, y_np = make_moons(n_samples=500, noise=0.25, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

plt.figure(figsize=(5, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", s=15)
plt.title("Conjunto de datos make_moons")
plt.show()

In [ ]:
class MLP(nn.Module):
    """Perceptrón multicapa (MLP) pequeño con dropout para regularizar."""

    def __init__(self, prob_dropout=0.3):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Dropout(prob_dropout),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(prob_dropout),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.capas(x)


def crear_modelo():
    """Crea el modelo con los mismos pesos iniciales en cada prueba,
    para que la comparación entre optimizadores sea justa."""
    torch.manual_seed(42)
    return MLP()

In [ ]:
def entrenar(optimizador_fn, n_epocas=150):
    modelo = crear_modelo()
    perdida_fn = nn.CrossEntropyLoss()
    optimizador = optimizador_fn(modelo.parameters())

    historial_perdida = []
    for _ in range(n_epocas):
        modelo.train()  # activa el dropout durante el entrenamiento
        optimizador.zero_grad()
        pred = modelo(X)
        perdida = perdida_fn(pred, y)
        perdida.backward()
        optimizador.step()
        historial_perdida.append(perdida.item())

    return historial_perdida

In [ ]:
tasa_aprendizaje = 0.05
decaimiento_pesos = ...  # TODO: define un valor típico de weight_decay (entre 1e-5 y 1e-3)

resultados = {
    "SGD (sin momento)": entrenar(
        lambda params: ...  # TODO: instancia torch.optim.SGD sin momento, con lr=tasa_aprendizaje y weight_decay=decaimiento_pesos
    ),
    "SGD con momento (0.9)": entrenar(
        lambda params: ...  # TODO: instancia torch.optim.SGD con momentum=0.9, lr=tasa_aprendizaje y weight_decay=decaimiento_pesos
    ),
    "Adam": entrenar(
        lambda params: ...  # TODO: instancia torch.optim.Adam con lr=tasa_aprendizaje y weight_decay=decaimiento_pesos
    ),
}

In [ ]:
plt.figure(figsize=(8, 5))
for nombre, perdidas in resultados.items():
    plt.plot(perdidas, label=nombre)

plt.xlabel("Época")
plt.ylabel("Pérdida (CrossEntropy)")
plt.title("Comparación de optimizadores en make_moons")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
tasa_aprendizaje = 0.05
decaimiento_pesos = 1e-4  # valore tipico: penalizza leggermente i pesi grandi

resultados = {
    "SGD (sin momento)": entrenar(
        lambda params: torch.optim.SGD(
            params, lr=tasa_aprendizaje, weight_decay=decaimiento_pesos
        )
    ),
    "SGD con momento (0.9)": entrenar(
        lambda params: torch.optim.SGD(
            params, lr=tasa_aprendizaje, momentum=0.9, weight_decay=decaimiento_pesos
        )
    ),
    "Adam": entrenar(
        lambda params: torch.optim.Adam(
            params, lr=tasa_aprendizaje, weight_decay=decaimiento_pesos
        )
    ),
}
```

Osserva il risultato: SGD senza momento è di gran lunga il più lento a far scendere la perdita nelle prime epoche. SGD con momento accelera questa discesa smussando la direzione di discesa. Adam di solito converge ancora più rapidamente all'inizio, perché adatta il tasso di apprendimento per parametro oltre ad accumulare momento, come hai visto in lezione.
</details>

## Per concludere

Hai già visto con i tuoi occhi perché il momento non è un dettaglio tecnico trascurabile: nella stessa valle stretta, la differenza tra usarlo o no si misura in ordini di grandezza di perdita finale. E in un problema di classificazione reale, scegliere bene tra SGD, SGD con momento e Adam, insieme a dropout e weight decay, fa la differenza tra aspettare ore o minuti per un modello addestrato.

**Torna ora al tuo progetto, cambia l'ottimizzatore del tuo modello attuale e verifica quante epoche ti servono per raggiungere la stessa perdita.**